<a href="https://colab.research.google.com/github/Andresg324/so101-vla-data-study/blob/main/so101_Data_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install "lerobot[smolvla,dataset]"

In [ ]:
!pip install av

In [ ]:
from huggingface_hub import login
login()

import wandb
wandb.login()

In [ ]:
from huggingface_hub import HfApi
for d in HfApi().list_datasets(author="Andresg324"):
    print(d.id)

In [ ]:
import subprocess

CONDITION = "randomized"
SEED      = 2000
DATASET   = "Andresg324/cube-pickup-randomized_20260809_115825"
OUTDIR    = f"outputs/train/smolvla_{CONDITION}_seed{SEED}"
MODELREPO = f"Andresg324/smolvla-cube-{CONDITION}-seed{SEED}"

subprocess.run(f"rm -rf {OUTDIR}", shell=True)

# Identical to the seed-1000 run except --seed and the output paths.
# Built as a Python string because IPython's {} expansion chokes on the
# JSON braces in --rename_map and silently passes the whole line through raw.
cmd = (
    "lerobot-train"
    " --policy.path=lerobot/smolvla_base"
    " --policy.push_to_hub=false"
    f" --dataset.repo_id={DATASET}"
    """ --rename_map='{"observation.images.overhead": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'"""
    " --batch_size=32"
    " --steps=10000"
    " --save_freq=2000"
    f" --seed={SEED}"
    f" --output_dir={OUTDIR}"
    f" --job_name=smolvla_{CONDITION}_seed{SEED}"
    " --policy.device=cuda"
    " --wandb.enable=true"
)
print(cmd)
subprocess.run(cmd, shell=True, check=True)     # ~50 min

api = HfApi()
api.create_repo(MODELREPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=f"{OUTDIR}/checkpoints/last/pretrained_model",
                  repo_id=MODELREPO, repo_type="model")
print(f"uploaded {MODELREPO}")

In [ ]:
#See where the checkpoint has landed
!ls {OUTDIR}/checkpoints/

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f"{MODELREPO}", repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=f"{OUTDIR}/checkpoints/last/pretrained_model",
    repo_id=f"{MODELREPO}",
    repo_type="model",
)